# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
bse = .002200

In [3]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

In [4]:
FINAL_OUTPUT_LOC = 'individual_corpora'
if not os.path.exists(FINAL_OUTPUT_LOC):
    os.mkdir(FINAL_OUTPUT_LOC)

In [5]:
# other global variables
turn_no_col = 'turn_delta'

In [6]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [7]:
# to rename the corpus correctly . . . 
def lang(x):
    result = str(x)
    
    if '-DISPEL-' in result:
        result = 'DISPEL'
    
    if result.startswith('GCSAusE'):
        result = 'GCSAusE'
    
    if result.startswith('se'):
        result = 'CORAAL'
        
    if result.startswith('call'):
        result = x.split('-')[0]
    
    if result.startswith('MICASE'):
        result = 'MICASE'
    
    if result.startswith('instruction'):
        result = x.split('-')[1]
    
    if result.startswith('CABNC'):
        result = 'CABNC'
        
    if result.startswith('SBCSAE'):
        result = 'SBCSAE'
        
    if result.startswith('SCoSE'):
        result = 'SCoSE'
        
    if result.startswith('candor'):
        result = 'CANDOR'
    
    if result.startswith('grace'):
        result = 'Miao-fNIRS'
    
    return result

#### Load data

In [8]:
df_ = pd.read_parquet(FINAL_DOC)

In [9]:
df_['corpus'] = [lang(co) for co in tqdm(df_['corpus'])]

  0%|          | 0/23233133 [00:00<?, ?it/s]

In [10]:
df_.head()

,Hxy,nx,ny,turn_delta,self,x_turn_id,x_speaker,corpus,conversation_id,sample_delta,resid
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,1,0.353162
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,2,0.034092
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,3,-0.381757
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,4,0.066321
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,5,0.207082


In [11]:
df_['corpus'].nunique()

14

In [12]:
df_['corpus'].value_counts()

array(['GCSAusE', 'SBCSAE', 'SCoSE', 'callfriend', 'callhome', 'DISPEL',
       'Frederiksen', 'Graesser', 'med_school', 'CANDOR', 'CABNC',
       'Miao-fNIRS', 'MICASE', 'CORAAL'], dtype=object)

## Vis & Other Tests

In [13]:
import seaborn as sns
import matplotlib.pyplot as plt

In [14]:
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [15]:
names = ['other', 'self']
main_line_colors = ['#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

### Individual conversations in a subplot

In [16]:
def timeplot(
        dataframe,
        x_data_col: str,
        y_data_col: str,
        groupby_cols: list = [],
        line_colors: list[tuple] = [],
        line_width: int = 3
):
    df_ = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'mean').reset_index()
    df_['std'] = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'std').reset_index()[y_data_col]
    df_['K'] = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'count').reset_index()[y_data_col]
    df_['se'] = df_['std'] / np.sqrt(df_['K'])
    # print(df_['se'])

    fig = go.Figure()

    for i, condset in enumerate(df_[groupby_cols].drop_duplicates().values):

        condset_name = '-'.join([str(cond) for cond in condset])

        # get where conditions are met
        sel = np.concatenate(
            [
                df_[groupby_cols[j]].isin([v]).values.reshape(-1, 1).astype(int)
                for j, v in enumerate(condset)
            ], axis=-1
        ).sum(axis=-1) == len(condset)

        if (len(line_colors) > 0) and (i < len(line_colors)):
            # get line colors
            line_color = 'rgba({}, {}, {}, 1.0)'.format(*line_colors[i])
            ribbon_color = 'rgba({}, {}, {}, 0.2)'.format(*line_colors[i])
        else:
            line_color = 'rgba(255, 127, 14, 1.0)'
            ribbon_color = 'rgba(255, 127, 14, 0.2)'

        # add mean values plot
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values,
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name,
                showlegend=True,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=line_width
                )
            )
        )

        # add ribbon upper bounds
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values + (2 * df_['se'].loc[sel].values),
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name + '_upperr_bound',
                showlegend=False,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=0
                )
            )
        )

        # add ribbon lower bounds
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values - (2 * df_['se'].loc[sel].values),
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name + '_lower_bound',
                showlegend=False,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=0
                ),
                fill='tonexty',
                fillcolor=ribbon_color
            )
        )

    return fig

#### Actually creating the figure

In [17]:
conversation_id_col = 'corpus'

In [18]:
k_conversations = df_['corpus'].nunique()
k_columns = 4
k_rows = int(np.ceil(k_conversations/k_columns))

In [19]:
assignments = sum([[(row+1,col+1) for col in range(k_columns)] for row in range(k_rows)], [])

### Specific Corpus Visualization

In [20]:
df_['corpus'].value_counts()

corpus
CORAAL         7322661
CABNC          5857285
MICASE         5017141
callhome       1479483
SBCSAE         1385635
callfriend      720347
Miao-fNIRS      634570
SCoSE           316582
CANDOR          281817
GCSAusE         127830
DISPEL           65145
Graesser         16830
med_school        5971
Frederiksen       1836
Name: count, dtype: int64

In [21]:
# SPECIFIC_CORPUS = 'GCSAusE'
# sel = df_['corpus'].isin([SPECIFIC_CORPUS])
# sel.sum()

In [22]:
corpora = df_['corpus'].unique().tolist()

In [23]:
# subplot_fig = make_subplots(
#     rows=k_rows, cols=k_columns,
#     subplot_titles=corpora + [None] * (k_conversations - len(corpora))
#     # row_titles=['Zoom', 'Telephone', 'Instruction', 'BNC/Naturalistic'],
#     # row_titles=['Instruction','Instruction', 'BNC/Naturalistic','BNC/Naturalistic'],
# )

In [24]:
for (i, conversation) in tqdm(enumerate(corpora)):
    df__ = df_.loc[
        df_[conversation_id_col].isin([conversation])
        & (df_[turn_no_col] > 0)
        & (df_[turn_no_col] <= 20)
    ].copy()
    df__['self'] = df__['self'].replace({True: 'self', False: 'other'})
    df__ = df__.sort_values(by=['self'])
    df__.index = range(len(df__))
    
    
    fig = timeplot(
        dataframe=df__,
        x_data_col='turn_delta',
        y_data_col='resid',
        groupby_cols=['self'],
        line_colors=[(65,105,225), (255, 127, 14)]
    ) 
    
    fig.update_layout(
        template='xgridoff',
        margin=dict(
            l=0,
            r=0,
            b=15,
            t=15
        )
    )
    fig.write_html(
        os.path.join(FINAL_OUTPUT_LOC, str(conversation)+'.html')
    )

0it [00:00, ?it/s]

In [25]:
# subplot_fig.update_layout(template='xgridoff')
# subplot_fig.show()

In [ ]:
# subplot_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta-individual-corpora.html'))